# REINFORCE and baselines: hands-on (optional)
**LLMs From Basics · Lesson RL.4**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

You'll implement a softmax policy and its score function, train it with **REINFORCE** on a bandit whose rewards are all positive, measure how much a **baseline** shrinks the variance of the gradient, and finally train a tiny 3-token "language model" with a sequence-level reward, the way LLM post-training does.

Stuck? Solutions are at the very bottom. Only `numpy` and `matplotlib` are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## 1 · A softmax policy
Write `softmax(logits)` (subtract the max first, for numerical safety) and `sample(logits, rng)`, which draws one action index with those probabilities.

In [ ]:
def softmax(logits):
    z = np.asarray(logits, dtype=float)
    # TODO: numerically stable softmax
    raise NotImplementedError("Exercise 1a")

def sample(logits, rng):
    # TODO: draw one index with probabilities softmax(logits)
    raise NotImplementedError("Exercise 1b")

**Check 1**

In [ ]:
assert np.allclose(softmax([0, 0, 0]), [1/3, 1/3, 1/3])
assert np.allclose(softmax([1000, 1000]), [0.5, 0.5]), "subtract the max to avoid overflow"
rng = np.random.default_rng(0)
counts = np.bincount([sample([np.log(0.7), np.log(0.2), np.log(0.1)], rng) for _ in range(20000)], minlength=3) / 20000
assert np.allclose(counts, [0.7, 0.2, 0.1], atol=0.015), counts
print("✓ exercise 1")

## 2 · The score function
For a softmax over logits $\theta$, the gradient of the log-probability of action $a$ is

$$\frac{\partial \log \pi_\theta(a)}{\partial \theta_b} = \mathbb{1}[b = a] - \pi_\theta(b)$$

Write `grad_log_pi(logits, a)` returning that vector. The check compares it with a finite-difference estimate and with the lesson's worked example.

In [ ]:
def grad_log_pi(logits, a):
    # TODO: one-hot(a) minus the probabilities
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
th = np.array([0.3, -1.2, 0.8])
num = np.array([(np.log(softmax(th + 1e-6 * np.eye(3)[b])[2]) - np.log(softmax(th - 1e-6 * np.eye(3)[b])[2])) / 2e-6 for b in range(3)])
assert np.allclose(grad_log_pi(th, 2), num, atol=1e-6)
# the lesson: theta = 0, sample Paris (index 0), G = +1, alpha = 0.5
th1 = np.zeros(3) + 0.5 * 1.0 * grad_log_pi(np.zeros(3), 0)
assert np.allclose(th1, [1/3, -1/6, -1/6]) and np.allclose(softmax(th1), [0.452, 0.274, 0.274], atol=1e-3)
# faded example: sample Lyon (index 1), G = -1
th2 = th1 + 0.5 * (-1.0) * grad_log_pi(th1, 1)
assert np.allclose(th2[1], -0.530, atol=1e-3) and np.allclose(softmax(th2)[0], 0.529, atol=1e-3)
assert abs(grad_log_pi(th, 0).sum()) < 1e-12, "the score function's components sum to zero"
print("✓ exercise 2")

## 3 · REINFORCE on a bandit, with and without a baseline
Three arms pay Gaussian rewards with means **5.0, 5.5 and 6.0** (std 1). Every reward is positive, which is exactly the case where plain REINFORCE struggles.

Write `reinforce_bandit(n_steps, alpha, use_baseline, seed)`. Each step: sample an arm, draw its reward $r$, and update

$$\theta \leftarrow \theta + \alpha\,(r - b)\,\nabla_\theta \log \pi_\theta(a)$$

with $b = 0$ if `use_baseline` is False, otherwise $b$ = the running average of all rewards seen **before** this step (start it at 0 and update it after the step: `b += (r - b) / t`). Return the history of $\pi_\theta(\text{best arm})$ after every step.

In [ ]:
MEANS = np.array([5.0, 5.5, 6.0])
def pull(a, rng):
    return MEANS[a] + rng.normal()

In [ ]:
def reinforce_bandit(n_steps=2000, alpha=0.01, use_baseline=False, seed=0):
    rng = np.random.default_rng(seed)
    theta, b, hist = np.zeros(3), 0.0, []
    for t in range(1, n_steps + 1):
        # TODO: sample, reward, REINFORCE update (with or without the running-average baseline)
        raise NotImplementedError("Exercise 3")
        hist.append(softmax(theta)[2])
    return np.array(hist)

**Check 3.** Averaged over 20 seeds, the baseline version should end up confidently on the best arm, and do better than the plain version.

In [ ]:
plain = np.mean([reinforce_bandit(use_baseline=False, seed=s) for s in range(20)], axis=0)
based = np.mean([reinforce_bandit(use_baseline=True, seed=s) for s in range(20)], axis=0)
print(f"P(best arm) after 2000 steps: plain {plain[-1]:.2f}, with baseline {based[-1]:.2f}")
assert based[-1] > 0.9
assert based[-1] > plain[-1]
plt.figure(figsize=(7, 3.5))
plt.plot(plain, label="REINFORCE, no baseline"); plt.plot(based, label="with running-average baseline")
plt.xlabel("step"); plt.ylabel("P(best arm), mean of 20 runs"); plt.legend(); plt.tight_layout(); plt.show()
print("✓ exercise 3")

## 4 · Measure the variance
The baseline doesn't change the gradient's *average*, only its *spread*. Write `grad_samples(logits, n, b, rng)` that returns an `(n, 3)` array of single-sample gradient estimates $(r - b)\,\nabla_\theta \log \pi_\theta(a)$, each with a freshly sampled arm and reward.

The check estimates the mean and variance at $\theta = 0$ with $b = 0$ and with $b = \mathbb{E}[r] = 5.5$.

In [ ]:
def grad_samples(logits, n, b, rng):
    # TODO: n single-sample REINFORCE gradient estimates, stacked into an (n, 3) array
    raise NotImplementedError("Exercise 4")

**Check 4**

In [ ]:
rng = np.random.default_rng(42)
g0 = grad_samples(np.zeros(3), 40000, 0.0, rng)
g1 = grad_samples(np.zeros(3), 40000, 5.5, rng)
true = np.array([(1/3) * (m - MEANS.mean()) for m in MEANS])   # exact gradient at theta = 0
print("mean, no baseline:  ", g0.mean(0).round(3))
print("mean, baseline 5.5: ", g1.mean(0).round(3), " exact:", true.round(3))
print("variance, no baseline:  ", g0.var(0).round(3))
print("variance, baseline 5.5: ", g1.var(0).round(3))
assert np.allclose(g1.mean(0), true, atol=0.02) and np.allclose(g0.mean(0), true, atol=0.08), "both are unbiased"
assert g1.var(0).sum() < g0.var(0).sum() / 10, "the baseline should cut the variance by more than 10x here"
print("✓ exercise 4")

## 5 · A tiny "LLM": rewards for whole sequences
Our toy language model writes a 3-token answer from a 4-token vocabulary. To keep it tiny, each position has its own logits (a `(3, 4)` array) and ignores the earlier tokens. The reward is **1 only if the whole sequence equals the target `[2, 0, 3]`**, else 0: one number for the whole response, like a verifier.

The log-probability of a sequence is the **sum** of its token log-probs, so its gradient is the per-position score functions stacked together. Write `seq_grad_log_prob(logits, seq)` returning a `(3, 4)` array, then `train_toy_lm`, which each step samples a group of `G` sequences, uses the **group-mean reward as the baseline** (as GRPO and RLOO do), and applies

$$\theta \leftarrow \theta + \frac{\alpha}{G}\sum_{i=1}^{G} (R_i - \bar R)\,\nabla_\theta \log \pi_\theta(y_i)$$

Return the probability of the target sequence after every step.

In [ ]:
TARGET = [2, 0, 3]
def seq_prob(logits, seq):
    return float(np.prod([softmax(logits[t])[tok] for t, tok in enumerate(seq)]))

In [ ]:
def seq_grad_log_prob(logits, seq):
    # TODO: row t = grad_log_pi(logits[t], seq[t])
    raise NotImplementedError("Exercise 5a")

def train_toy_lm(n_steps=300, G=16, alpha=1.0, seed=0):
    rng = np.random.default_rng(seed)
    logits, hist = np.zeros((3, 4)), []
    for step in range(n_steps):
        # TODO: sample G sequences, reward 1 for an exact match, group-mean baseline, update
        raise NotImplementedError("Exercise 5b")
        hist.append(seq_prob(logits, TARGET))
    return np.array(hist)

**Check 5.** At the start the target has probability $(1/4)^3 = 1/64$; most groups contain no success at all, so they teach nothing (every advantage is 0). Once a success appears, every token of it is pushed up together.

In [ ]:
assert np.isclose(seq_prob(np.zeros((3, 4)), TARGET), 1 / 64)
g = seq_grad_log_prob(np.zeros((3, 4)), TARGET)
assert g.shape == (3, 4) and np.allclose(g[0], [-0.25, -0.25, 0.75, -0.25])
runs = [train_toy_lm(seed=s) for s in range(5)]
finals = [r[-1] for r in runs]
print("P(target) after 300 steps:", np.round(finals, 3))
assert np.median(finals) > 0.8
plt.figure(figsize=(7, 3.5))
for r in runs: plt.plot(r)
plt.xlabel("step"); plt.ylabel("P(target sequence)"); plt.tight_layout(); plt.show()
print("✓ exercise 5")

## Experiment
1. In exercise 3, try `alpha = 0.05` without a baseline. What happens to individual runs (plot a few seeds instead of the mean)?
2. Change the bandit means to **−0.5, 0, +0.5**. Does the baseline still matter as much? Why?
3. In exercise 5, set the baseline to 0 (use `R` instead of `R - R.mean()`). With 0/1 rewards that's still a valid gradient. Is it slower or faster here, and why might group baselines matter more when rewards are like 0.9 vs 1.0?
4. Try `G = 4` and `G = 64`. What does the group size do to the number of "wasted" steps at the start?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for G in [4, 16, 64]:
    ax.plot(np.mean([train_toy_lm(G=G, seed=s) for s in range(5)], axis=0), label=f"G = {G}")
ax.set_xlabel("step"); ax.set_ylabel("P(target), mean of 5 runs"); ax.legend(); plt.tight_layout(); plt.show()

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def softmax(logits):
    z = np.asarray(logits, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()

def sample(logits, rng):
    return int(rng.choice(len(logits), p=softmax(logits)))

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def grad_log_pi(logits, a):
    g = -softmax(logits)
    g[a] += 1.0
    return g

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def reinforce_bandit(n_steps=2000, alpha=0.01, use_baseline=False, seed=0):
    rng = np.random.default_rng(seed)
    theta, b, hist = np.zeros(3), 0.0, []
    for t in range(1, n_steps + 1):
        a = sample(theta, rng)
        r = pull(a, rng)
        adv = r - b if use_baseline else r
        theta = theta + alpha * adv * grad_log_pi(theta, a)
        b += (r - b) / t
        hist.append(softmax(theta)[2])
    return np.array(hist)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def grad_samples(logits, n, b, rng):
    out = np.empty((n, len(logits)))
    for i in range(n):
        a = sample(logits, rng)
        r = pull(a, rng)
        out[i] = (r - b) * grad_log_pi(logits, a)
    return out

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def seq_grad_log_prob(logits, seq):
    return np.stack([grad_log_pi(logits[t], tok) for t, tok in enumerate(seq)])

def train_toy_lm(n_steps=300, G=16, alpha=1.0, seed=0):
    rng = np.random.default_rng(seed)
    logits, hist = np.zeros((3, 4)), []
    for step in range(n_steps):
        seqs = [[sample(logits[t], rng) for t in range(3)] for _ in range(G)]
        R = np.array([1.0 if s == TARGET else 0.0 for s in seqs])
        adv = R - R.mean()
        grad = sum(adv[i] * seq_grad_log_prob(logits, seqs[i]) for i in range(G)) / G
        logits = logits + alpha * grad
        hist.append(seq_prob(logits, TARGET))
    return np.array(hist)